In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score
)

# Load data
df = pd.read_csv("student_lifestyle_100k.csv")

X = df.drop(columns=["Depression", "Student_ID"])
y = df["Depression"].astype(int)

# Train / validation / test split
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.1765,
    random_state=42,
    stratify=y_temp
)

# Preprocessing
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

# Model
gb_model = GradientBoostingClassifier(
    n_estimators=250,
    learning_rate=0.05,
    max_depth=3,
    min_samples_leaf=20,
    subsample=0.85,
    random_state=42
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", gb_model)
    ]
)

# Train
pipeline.fit(X_train, y_train)

# Tune threshold using validation set
val_probs = pipeline.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.05, 0.95, 0.01)

best_threshold = 0.5
best_f1 = 0

for threshold in thresholds:
    val_preds = (val_probs >= threshold).astype(int)
    score = f1_score(y_val, val_preds)

    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

print("Best threshold:", best_threshold)
print("Best validation F1:", best_f1)


# Test evaluation
test_probs = pipeline.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

cm = confusion_matrix(y_test, test_preds)
tn, fp, fn, tp = cm.ravel()

specificity = tn / (tn + fp)
auc_roc = roc_auc_score(y_test, test_probs)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, test_preds))

print("Accuracy:", accuracy_score(y_test, test_preds))
print("Precision:", precision_score(y_test, test_preds))
print("Recall:", recall_score(y_test, test_preds))
print("Specificity:", specificity)
print("F1 Score:", f1_score(y_test, test_preds))
print("AUC-ROC:", auc_roc)

Best threshold: 0.17000000000000004
Best validation F1: 0.3313180169286578
Confusion Matrix:
[[10218  3273]
 [  595   914]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.76      0.84     13491
           1       0.22      0.61      0.32      1509

    accuracy                           0.74     15000
   macro avg       0.58      0.68      0.58     15000
weighted avg       0.87      0.74      0.79     15000

Accuracy: 0.7421333333333333
Precision: 0.21829472175782183
Recall: 0.6056991385023194
Specificity: 0.7573938181009562
F1 Score: 0.32092696629213485
AUC-ROC: 0.6999646181910834
